In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling for report-ready figures
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
sns.set_palette("deep")

DATA_DIR = Path("data")
parquet_files = sorted(DATA_DIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} total parquet files under {DATA_DIR}/")

# Filter out tiny reconnect fragments (< 500 rows) and tag each valid file with a chunk_id
MIN_ROWS = 500
valid_dfs = []
skipped_fragments = 0

for chunk_id, fp in enumerate(parquet_files):
    df_chunk = pl.read_parquet(fp)
    if df_chunk.height < MIN_ROWS:
        skipped_fragments += 1
        continue
    
    # Extract base contract name (CONTROLS-2026, SENATETX-26, SENATEGA-26) and party (-D or -R)
    df_chunk = df_chunk.with_columns([
        pl.lit(chunk_id).alias("chunk_id"),
        pl.lit(fp.stem).alias("file_stem"),
        pl.col("ticker").cast(pl.Utf8).str.replace(r"-[DR]$", "").alias("contract"),
        pl.col("ticker").cast(pl.Utf8).str.extract(r"-([DR])$", 1).alias("party"),
    ])
    valid_dfs.append(df_chunk)

df = pl.concat(valid_dfs, how="vertical_relaxed")
print(f"Loaded {len(valid_dfs)} valid continuous chunks (skipped {skipped_fragments} short reconnect fragments < {MIN_ROWS} rows).")
print(f"Total rows loaded: {df.height:,}\n")

# Summary table by Contract and Party Book
inventory = (
    df.group_by(["contract", "party", "ticker"])
    .agg([
        pl.col("chunk_id").n_unique().alias("num_chunks"),
        pl.len().alias("total_events"),
        (pl.col("timestamp_ns").min() / 1e9).cast(pl.Int64).alias("min_epoch_s"),
        (pl.col("timestamp_ns").max() / 1e9).cast(pl.Int64).alias("max_epoch_s"),
    ])
    .sort(["contract", "party"])
)
print(inventory)

In [ ]:
# Check 1: Event Type Composition & Trade Density per Book
event_comp = (
    df.group_by(["contract", "ticker"])
    .agg([
        pl.len().alias("total_events"),
        (pl.col("event_type") == "trade").sum().alias("n_trades"),
        (pl.col("event_type") == "delta_book").sum().alias("n_delta_book"),
        (pl.col("event_type") == "snapshot").sum().alias("n_snapshots"),
        ((pl.col("event_type") == "trade") & (pl.col("taker_side") == "yes_taker")).sum().alias("n_yes_trades"),
        ((pl.col("event_type") == "trade") & (pl.col("taker_side") == "no_taker")).sum().alias("n_no_trades"),
        pl.col("trade_size").filter(pl.col("event_type") == "trade").sum().alias("total_trade_vol"),
    ])
    .with_columns([
        (pl.col("n_trades") / pl.col("total_events") * 100).round(2).alias("trade_pct (%)"),
        (pl.col("total_events") / pl.col("n_trades")).round(1).alias("events_per_trade"),
    ])
    .sort(["contract", "ticker"])
)

print("=== 1. Event Composition & Trade Density ===")
print(event_comp.select([
    "ticker", "total_events", "n_trades", "trade_pct (%)", 
    "events_per_trade", "n_yes_trades", "n_no_trades", "total_trade_vol"
]))

# Check expected number of trades inside candidate forward horizons H in {20, 50, 100, 200}
print("\n=== Expected Number of Trades Inside Candidate Forward Horizons H ===")
h_check = event_comp.select([
    "ticker",
    "events_per_trade",
    (20 / pl.col("events_per_trade")).round(2).alias("E[trades] @ H=20"),
    (50 / pl.col("events_per_trade")).round(2).alias("E[trades] @ H=50"),
    (100 / pl.col("events_per_trade")).round(2).alias("E[trades] @ H=100"),
    (200 / pl.col("events_per_trade")).round(2).alias("E[trades] @ H=200"),
])
print(h_check)

In [ ]:
# Check 2A: Distribution of single-step physical waiting time delta_t_ms (excluding the first row of each chunk)
dt_stats = (
    df.filter(pl.col("delta_t_ms").is_not_null() & (pl.col("event_seq") > 0))
    .group_by("ticker")
    .agg([
        pl.col("delta_t_ms").quantile(0.10).round(2).alias("p10_ms"),
        pl.col("delta_t_ms").quantile(0.50).round(2).alias("median_ms"),
        pl.col("delta_t_ms").mean().round(2).alias("mean_ms"),
        pl.col("delta_t_ms").quantile(0.90).round(2).alias("p90_ms"),
        pl.col("delta_t_ms").quantile(0.99).round(2).alias("p99_ms"),
    ])
    .sort("ticker")
)
print("=== 2A. Inter-Event Physical Duration (delta_t_ms) Quantiles ===")
print(dt_stats)

# Check 2B: Wall-clock duration (in minutes) spanned by W events within each continuous chunk
candidate_W = [100, 500, 1000, 2000]
w_exprs = [
    ((pl.col("timestamp_ns").shift(-w) - pl.col("timestamp_ns")).over("chunk_id") / (1e9 * 60.0)).alias(f"span_W{w}_min")
    for w in candidate_W
]

df_spans = df.with_columns(w_exprs)

span_summary = (
    df_spans.group_by("ticker")
    .agg([
        pl.col("span_W100_min").median().round(2).alias("W=100 (med_min)"),
        pl.col("span_W500_min").median().round(2).alias("W=500 (med_min)"),
        pl.col("span_W500_min").quantile(0.90).round(2).alias("W=500 (p90_min)"),
        pl.col("span_W1000_min").median().round(2).alias("W=1000 (med_min)"),
        pl.col("span_W2000_min").median().round(2).alias("W=2000 (med_min)"),
        pl.col("span_W2000_min").quantile(0.90).round(2).alias("W=2000 (p90_min)"),
    ])
    .sort("ticker")
)
print("\n=== 2B. Wall-Clock Time (in Minutes) Spanned by Lookback Window W ===")
print(span_summary)

In [ ]:
# Compute Mid-Price, Log-Odds Mid, L1/L5 Queue Depth, and Total Depth from the nested lists
PRICE_SCALE = 1000.0  # Your process_data.py scales prices by 1000 (0..1000 ticks)

df_micro = (
    df.filter(pl.col("best_bid").is_not_null() & pl.col("best_ask").is_not_null())
    .with_columns([
        ((pl.col("best_bid") + pl.col("best_ask")) / (2.0 * PRICE_SCALE)).alias("mid_prob"),
        (pl.col("spread") / PRICE_SCALE * 100.0).alias("spread_cents"),
        pl.col("bid_sizes").list.head(1).list.sum().alias("bid_l1_depth"),
        pl.col("ask_sizes").list.head(1).list.sum().alias("ask_l1_depth"),
        pl.col("bid_sizes").list.head(5).list.sum().alias("bid_l5_depth"),
        pl.col("ask_sizes").list.head(5).list.sum().alias("ask_l5_depth"),
    ])
    .with_columns([
        (pl.col("mid_prob").clip(0.001, 0.999)).alias("mid_prob_clipped"),
        (pl.col("bid_l5_depth") + pl.col("ask_l5_depth")).alias("total_l5_depth"),
    ])
    .with_columns([
        (pl.col("mid_prob_clipped") / (1.0 - pl.col("mid_prob_clipped"))).log().alias("log_odds_mid")
    ])
)

# Print summary table of microstructure regime metrics
regime_table = (
    df_micro.group_by(["contract", "ticker"])
    .agg([
        pl.col("mid_prob").median().round(3).alias("med_mid_price ($)"),
        pl.col("log_odds_mid").median().round(2).alias("med_log_odds"),
        pl.col("spread_cents").median().round(2).alias("med_spread (¢)"),
        pl.col("spread_cents").mean().round(2).alias("mean_spread (¢)"),
        (pl.col("bid_l1_depth") + pl.col("ask_l1_depth")).median().alias("med_L1_depth"),
        pl.col("total_l5_depth").median().alias("med_L5_depth"),
    ])
    .sort(["contract", "ticker"])
)
print("=== 3A. Microstructure Regime Summary Across Contracts ===")
print(regime_table)

# Trade size quantiles for determining V_min
trade_quantiles = (
    df.filter(pl.col("event_type") == "trade")
    .group_by(["contract", "ticker"])
    .agg([
        pl.len().alias("n_trades"),
        pl.col("trade_size").quantile(0.50).alias("p50_size"),
        pl.col("trade_size").quantile(0.75).alias("p75_size"),
        pl.col("trade_size").quantile(0.90).alias("p90_size"),
        pl.col("trade_size").quantile(0.95).alias("p95_size"),
        pl.col("trade_size").quantile(0.99).alias("p99_size"),
        pl.col("trade_size").max().alias("max_size"),
    ])
    .sort(["contract", "ticker"])
)
print("\n=== 3B. Single-Trade Volume Quantiles (Guides V_min) ===")
print(trade_quantiles)

# 4-Panel Plot for Report
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
contracts = ["CONTROLS-2026", "SENATETX-26", "SENATEGA-26"]
colors = {"CONTROLS-2026": "#1f77b4", "SENATETX-26": "#2ca02c", "SENATEGA-26": "#d62728"}

# Panel 1: Raw Mid-Price Distribution (Focusing on -D book for consistent party orientation, or all)
for c in contracts:
    sub = df_micro.filter((pl.col("contract") == c) & (pl.col("party") == "D"))["mid_prob"].to_numpy()
    axes[0, 0].hist(sub, bins=60, alpha=0.55, density=True, label=f"{c}-D", color=colors[c])
axes[0, 0].set_title("1. Implied Probability ($P_{mid}$) — Interior vs. Boundary Regime")
axes[0, 0].set_xlabel("Mid Price ($)")
axes[0, 0].set_ylabel("Density")
axes[0, 0].legend()

# Panel 2: Log-Odds Transformed Price
for c in contracts:
    sub = df_micro.filter((pl.col("contract") == c) & (pl.col("party") == "D"))["log_odds_mid"].to_numpy()
    axes[0, 1].hist(sub, bins=60, alpha=0.55, density=True, label=f"{c}-D", color=colors[c])
axes[0, 1].set_title("2. Log-Odds Mid Price $\\ln(P_{mid} / (1 - P_{mid}))$")
axes[0, 1].set_xlabel("Log-Odds Mid")
axes[0, 1].set_ylabel("Density")
axes[0, 1].legend()

# Panel 3: Log10(L5 Total Queue Depth) Comparison
for c in contracts:
    sub = np.log10(df_micro.filter(pl.col("contract") == c)["total_l5_depth"].to_numpy() + 1.0)
    axes[1, 0].hist(sub, bins=50, alpha=0.55, density=True, label=c, color=colors[c])
axes[1, 0].set_title("3. Top-5 Queue Depth Distribution $\\log_{10}(\\text{Depth}_{L5} + 1)$")
axes[1, 0].set_xlabel("$\\log_{10}(\\text{Contracts})$")
axes[1, 0].set_ylabel("Density")
axes[1, 0].legend()

# Panel 4: Trade Size Empirical CCDF (Log-Log Tail Plot)
for c in contracts:
    sizes = np.sort(df.filter((pl.col("contract") == c) & (pl.col("event_type") == "trade"))["trade_size"].to_numpy())
    if len(sizes) > 0:
        ccdf = 1.0 - np.arange(1, len(sizes) + 1) / len(sizes)
        axes[1, 1].loglog(sizes, ccdf, label=f"{c} (n={len(sizes):,})", color=colors[c], lw=2)
axes[1, 1].set_title("4. Trade Size Tail Distribution (Empirical CCDF, Log-Log)")
axes[1, 1].set_xlabel("Trade Size (Contracts)")
axes[1, 1].set_ylabel("$P(\\text{Trade Size} > x)$")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Check 4: Inspect exact timestamp gaps between each 'trade' and the immediately preceding/succeeding 'delta_book'
df_order_check = (
    df.with_columns([
        pl.col("event_type").shift(1).over("chunk_id").alias("prev_type"),
        pl.col("delta_t_ms").alias("dt_from_prev_ms"),
        pl.col("event_type").shift(-1).over("chunk_id").alias("next_type"),
        pl.col("delta_t_ms").shift(-1).over("chunk_id").alias("dt_to_next_ms"),
        pl.col("bid_sizes").list.head(1).list.sum().alias("bid_l1"),
        pl.col("ask_sizes").list.head(1).list.sum().alias("ask_l1"),
    ])
    .with_columns([
        (pl.col("bid_l1") - pl.col("bid_l1").shift(1).over("chunk_id")).alias("d_bid_l1_at_step"),
        (pl.col("ask_l1") - pl.col("ask_l1").shift(1).over("chunk_id")).alias("d_ask_l1_at_step"),
    ])
)

trades_only = df_order_check.filter(pl.col("event_type") == "trade")

timing_summary = (
    trades_only.group_by("contract")
    .agg([
        pl.len().alias("total_trades"),
        # How often does a delta_book happen within <= 5ms BEFORE the trade print?
        ((pl.col("prev_type") == "delta_book") & (pl.col("dt_from_prev_ms") == 0)).sum().alias("prev_delta_0ms"),
        ((pl.col("prev_type") == "delta_book") & (pl.col("dt_from_prev_ms") > 0) & (pl.col("dt_from_prev_ms") <= 5.0)).sum().alias("prev_delta_0_to_5ms"),
        ((pl.col("prev_type") == "delta_book") & (pl.col("dt_from_prev_ms") > 5.0) & (pl.col("dt_from_prev_ms") <= 50.0)).sum().alias("prev_delta_5_to_50ms"),
        # How often does a delta_book happen within <= 5ms AFTER the trade print?
        ((pl.col("next_type") == "delta_book") & (pl.col("dt_to_next_ms") == 0)).sum().alias("next_delta_0ms"),
        ((pl.col("next_type") == "delta_book") & (pl.col("dt_to_next_ms") > 0) & (pl.col("dt_to_next_ms") <= 5.0)).sum().alias("next_delta_0_to_5ms"),
        ((pl.col("next_type") == "delta_book") & (pl.col("dt_to_next_ms") > 5.0) & (pl.col("dt_to_next_ms") <= 50.0)).sum().alias("next_delta_5_to_50ms"),
    ])
    .sort("contract")
)
print("=== 4A. Micro-Timestamp Ordering: Is L2 Depth Depleted BEFORE or AFTER the Trade Print? ===")
print(timing_summary)

# Print a 7-row window around a large trade (>= 200 contracts) on CONTROLS-2026 to see the raw sequence
sample_large_trade = (
    df_order_check.filter(
        (pl.col("contract") == "CONTROLS-2026") & 
        (pl.col("event_type") == "trade") & 
        (pl.col("trade_size") >= 200)
    )
    .head(1)
)

if sample_large_trade.height > 0:
    target_chunk = sample_large_trade["chunk_id"][0]
    target_seq = sample_large_trade["event_seq"][0]
    print(f"\n=== 4B. Sample Micro-Tape Around a Large Trade (chunk={target_chunk}, event_seq={target_seq}) ===")
    tape_window = (
        df_order_check.filter(
            (pl.col("chunk_id") == target_chunk) & 
            (pl.col("event_seq").is_between(target_seq - 3, target_seq + 3))
        )
        .select([
            "event_seq", "event_type", "delta_t_ms", "taker_side", "trade_price", 
            "trade_size", "best_bid", "best_ask", "bid_l1", "ask_l1"
        ])
    )
    print(tape_window)

In [ ]:
# Check 5: Cross-Book Lead-Lag Analysis between Economically Equivalent Pairs:
# Pair 1 (Pro-Republican): R_yes  <-->  D_no
# Pair 2 (Pro-Democrat):   D_yes  <-->  R_no

LARGE_TRADE_FLOOR = 50  # Minimum trade size to consider an "initiator" impulse

trades_all = (
    df.filter(pl.col("event_type") == "trade")
    .select(["contract", "party", "timestamp_ns", "taker_side", "trade_size"])
    .sort("timestamp_ns")
)

def analyze_cross_book_lag(contract_name: str, min_size: int = LARGE_TRADE_FLOOR):
    sub = trades_all.filter(pl.col("contract") == contract_name)
    
    # Split into the 4 directional trade streams
    r_yes = sub.filter((pl.col("party") == "R") & (pl.col("taker_side") == "yes_taker")).sort("timestamp_ns")
    d_no  = sub.filter((pl.col("party") == "D") & (pl.col("taker_side") == "no_taker")).sort("timestamp_ns")
    d_yes = sub.filter((pl.col("party") == "D") & (pl.col("taker_side") == "yes_taker")).sort("timestamp_ns")
    r_no  = sub.filter((pl.col("party") == "R") & (pl.col("taker_side") == "no_taker")).sort("timestamp_ns")
    
    # Direction A: Large R_yes -> how soon does next D_no occur?
    r_yes_large = r_yes.filter(pl.col("trade_size") >= min_size)
    d_yes_large = d_yes.filter(pl.col("trade_size") >= min_size)
    d_no_large  = d_no.filter(pl.col("trade_size") >= min_size)
    r_no_large  = r_no.filter(pl.col("trade_size") >= min_size)

    pairs = [
        ("R_yes -> D_no", r_yes_large, d_no),
        ("D_no  -> R_yes", d_no_large, r_yes),
        ("D_yes -> R_no", d_yes_large, r_no),
        ("R_no  -> D_yes", r_no_large, d_yes),
    ]
    
    rows = []
    for label, leader_df, follower_df in pairs:
        if leader_df.height == 0 or follower_df.height == 0:
            continue
        
        # Find the very next trade on the paired book strictly after leader timestamp
        joined = leader_df.join_asof(
            follower_df.select([
                pl.col("timestamp_ns").alias("follower_ts_ns"),
                pl.col("trade_size").alias("follower_size")
            ]),
            left_on="timestamp_ns",
            right_on="follower_ts_ns",
            strategy="forward"
        ).with_columns([
            ((pl.col("follower_ts_ns") - pl.col("timestamp_ns")) / 1e9).alias("lag_sec")
        ])
        
        n_total = joined.height
        rows.append({
            "contract": contract_name,
            "pair": label,
            "n_leader_trades": n_total,
            "prob_<=0.5s (%)": round(joined.filter(pl.col("lag_sec") <= 0.5).height / n_total * 100, 1),
            "prob_<=1.0s (%)": round(joined.filter(pl.col("lag_sec") <= 1.0).height / n_total * 100, 1),
            "prob_<=3.0s (%)": round(joined.filter(pl.col("lag_sec") <= 3.0).height / n_total * 100, 1),
            "prob_<=5.0s (%)": round(joined.filter(pl.col("lag_sec") <= 5.0).height / n_total * 100, 1),
            "prob_<=10.0s (%)": round(joined.filter(pl.col("lag_sec") <= 10.0).height / n_total * 100, 1),
            "median_lag (s)": round(joined["lag_sec"].median(), 2) if joined["lag_sec"].drop_nulls().len() > 0 else None,
        })
    return pl.DataFrame(rows)

lag_tables = [analyze_cross_book_lag(c) for c in ["CONTROLS-2026", "SENATETX-26", "SENATEGA-26"]]
lag_summary = pl.concat([t for t in lag_tables if t.height > 0])
print(f"=== 5. Cross-Book Lead-Lag Response Probability (Leader Trade >= {LARGE_TRADE_FLOOR} Contracts) ===")
print(lag_summary)